# ClaimLens — Phase 1 baseline on Kaggle (free T4 GPU)

Runs an **un-tuned** open vision-language model (Qwen2.5-VL-3B-Instruct) over the synthetic test set and scores it.
This is the **"before" number** that SFT (Phase 2) and RL (Phase 3) must beat.

### One-time setup
1. Kaggle → **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. Right-hand panel → **Settings**:
   - **Accelerator:** `GPU T4 x2` (or `GPU T4 x1`)
   - **Internet:** On (needs a phone-verified account)
3. Get the project code into the notebook — pick one:
   - **A. GitHub (recommended):** push the `claimLens` folder to a GitHub repo and set `REPO_URL` below.
   - **B. Upload:** zip the `claimLens` folder, then **Add Input → Upload → New Dataset**, name it `claimlens-code`.

### Budget
Roughly 20–60 seconds per document on a T4 (it depends on output length). Start with `LIMIT = 10`, check the output
looks sane, then do the full run. Use **Save Version → Save & Run All (Commit)** for long runs — it keeps running with
your browser closed and saves `/kaggle/working` as output.

In [ ]:
REPO_URL = ""          # e.g. "https://github.com/<you>/claimLens.git"  (option A)
UPLOADED = "/kaggle/input"   # option B: searched recursively, so the exact dataset path does not matter
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LIMIT = 10             # set to None for the full test split
LOAD_IN_4BIT = False   # True if you run out of GPU memory or get garbage output in fp16

In [ ]:
import os, shutil, subprocess, sys, glob
os.chdir("/kaggle/working")
if REPO_URL:
    if not os.path.exists("claimLens"):
        subprocess.run(["git", "clone", REPO_URL, "claimLens"], check=True)
else:
    hits = glob.glob(f"{UPLOADED}/**/claimlens/__init__.py", recursive=True)
    assert hits, "Project not found -- set REPO_URL or check the uploaded dataset path"
    src = os.path.dirname(os.path.dirname(hits[0]))   # project root = parent of the package folder
    shutil.copytree(src, "claimLens", dirs_exist_ok=True)
os.chdir("/kaggle/working/claimLens")
sys.path.insert(0, os.getcwd())
print(os.getcwd(), os.listdir("."))

In [ ]:
# Kaggle images already include torch; add/upgrade the rest.
!pip install -q -U "transformers>=4.49" accelerate "pydantic>=2" bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Generate the dataset
Generating here (not on your laptop) keeps fonts identical between runs — different fonts mean a different dataset hash.

In [ ]:
!python -m claimlens generate --out data --train 300 --val 50 --test 100 --fair-sets 25 --fair-variants 4

## 2. Load the model once

In [ ]:
from claimlens.predictors import HFVisionPredictor
from claimlens.runner import run_predictions, evaluate_run

pred = HFVisionPredictor(MODEL_ID, load_in_4bit=LOAD_IN_4BIT, max_new_tokens=900)
print(pred.model_id, pred.model_revision, "| prompt", pred.prompt_version)

## 3. Sanity check on one document
Read the raw output. Is it JSON? Are values plausible? Are bboxes numbers in the right range (0–980 wide, 0–1260 tall)?
If you see gibberish or `!!!!`, restart with `LOAD_IN_4BIT = True`.

In [ ]:
import json
from pathlib import Path
label = json.loads(Path("data/test/labels/test-00000.json").read_text())
out = pred.predict(Path("data/test") / label["image"], label)
print(out["latency_ms"], "ms,", out["output_tokens"], "output tokens\n")
print(out["raw_text"][:3000])

## 4. Baseline runs: test split
Each experiment gets its own run folder, so results stay comparable and auditable.
You can change the prompt version and decoding on the same loaded model without reloading it.

In [ ]:
def run_experiment(prompt_version, finish_json, split="test", limit=LIMIT):
    pred.prompt_version, pred.finish_json = prompt_version, finish_json
    pred.decoding = "greedy+finish_json" if finish_json else "greedy"
    run_id = f"baseline_{prompt_version}_{pred.decoding.replace('+', '-')}_{split}"
    run_dir = run_predictions(pred, "data", split, runs_dir="runs", run_id=run_id, limit=limit)
    evaluate_run(run_dir)
    print(open(f"{run_dir}/report.md").read())
    return run_dir

# Experiment A: improved prompt (v2), plain decoding
run_a = run_experiment("v2", finish_json=False)

In [ ]:
# Experiment B: v2 prompt + 'finish_json' (the model is not allowed to stop until its JSON is closed)
run_b = run_experiment("v2", finish_json=True)

## 5. Compare all runs side by side

In [ ]:
import json, glob
import pandas as pd
rows = []
for mpath in sorted(glob.glob("runs/*/metrics.json")):
    m = json.load(open(mpath)); cfg = json.load(open(mpath.replace("metrics.json", "config.json"))); s = m["summary"]
    rows.append({"run": m["run_id"], "prompt": cfg["prompt_version"], "decoding": cfg.get("decoding", "greedy"),
                 "docs": s["n_docs"], "valid_json": s["json_valid_rate"], "field_acc": s["field_accuracy"],
                 "grounded_acc": s["grounded_accuracy"], "hallucination": s["hallucination_rate"],
                 "reward": s["mean_reward"], "p50_ms": s.get("latency_ms_p50")})
pd.DataFrame(rows)

## 6. Fairness run: counterfactual split
Same documents, only the claimant name changes. Skip this until the test run looks sane.

In [ ]:
RUN_FAIRNESS = False
if RUN_FAIRNESS:
    run_experiment("v2", finish_json=True, split="fairness")   # use whichever setup won in section 5

## 7. Save results
Download `results.zip` from the **Output** panel and unzip it into your local `claimLens` folder (it contains
`runs/` plus the test and fairness images, so the error-analysis picture gallery works). Then open
`notebooks/02_error_analysis.ipynb` on your laptop and point `RUN_DIR` at the baseline run.

In [ ]:
!cd /kaggle/working/claimLens && zip -qr /kaggle/working/results.zip runs data/dataset_info.json data/test data/fairness && ls -lh /kaggle/working/results.zip